### 1. LIBRARIES

In [16]:
#For data analysis and processing
import pandas as pd 

### 2. DATA LOADING AND INITIAL AUDIT

In [17]:
#Loading all five datasets
accounts = pd.read_csv("ravenstack_accounts.csv")
subscriptions = pd.read_csv("ravenstack_subscriptions.csv")
support = pd.read_csv("ravenstack_support_tickets.csv")
churn = pd.read_csv("ravenstack_churn_events.csv")
usage = pd.read_csv("ravenstack_feature_usage.csv")

#Storing datasets in a dictionary for easier auditing
datasets = {
    "accounts": accounts,
    "subscriptions": subscriptions,
    "support": support,
    "churn": churn,
    "usage": usage
}

#Printing the structure, columns, missing values, and duplicates for each dataset
for name, df in datasets.items():
    print(f"\n--- {name.upper()} ---")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print("\nMissing values:")
    print(df.isna().sum())
    print("\nDuplicate rows:", df.duplicated().sum())


--- ACCOUNTS ---
Shape: (500, 10)
Columns: ['account_id', 'account_name', 'industry', 'country', 'signup_date', 'referral_source', 'plan_tier', 'seats', 'is_trial', 'churn_flag']

Missing values:
account_id         0
account_name       0
industry           0
country            0
signup_date        0
referral_source    0
plan_tier          0
seats              0
is_trial           0
churn_flag         0
dtype: int64

Duplicate rows: 0

--- SUBSCRIPTIONS ---
Shape: (5000, 14)
Columns: ['subscription_id', 'account_id', 'start_date', 'end_date', 'plan_tier', 'seats', 'mrr_amount', 'arr_amount', 'is_trial', 'upgrade_flag', 'downgrade_flag', 'churn_flag', 'billing_frequency', 'auto_renew_flag']

Missing values:
subscription_id         0
account_id              0
start_date              0
end_date             4514
plan_tier               0
seats                   0
mrr_amount              0
arr_amount              0
is_trial                0
upgrade_flag            0
downgrade_flag          

### 3. DATA CLEANING AND TYPE FIXING

In [18]:
#DATE-TIME FORMATTING

#Converting account signup_date column into datetime format
accounts["signup_date"] = pd.to_datetime(accounts["signup_date"], errors = "coerce") 

#Converting subscriptions start_date and end_date columns into datetime format
subscriptions["start_date"] = pd.to_datetime(subscriptions["start_date"], errors = "coerce")
subscriptions["end_date"] = pd.to_datetime(subscriptions["end_date"], errors = "coerce")

#Converting support submitted_at and closed_at columns into datetime format
support["submitted_at"] = pd.to_datetime(support["submitted_at"], errors = "coerce")
support["closed_at"] = pd.to_datetime(support["closed_at"], errors = "coerce")

#Converting churn churn_date column into datetime format
churn["churn_date"] = pd.to_datetime(churn["churn_date"], errors = "coerce") 
#Converting usage usage_date columns into datetime format
usage["usage_date"] = pd.to_datetime(usage["usage_date"], errors = "coerce") 

#Forcing important numeric columns to be numeric
subscriptions["mrr_amount"] = pd.to_numeric(subscriptions["mrr_amount"], errors = "coerce")
subscriptions["arr_amount"] = pd.to_numeric(subscriptions["arr_amount"], errors = "coerce")

support["resolution_time_hours"] = pd.to_numeric(support["resolution_time_hours"], errors = "coerce")
support["first_response_time_minutes"] = pd.to_numeric(support["first_response_time_minutes"], errors = "coerce")
support["satisfaction_score"] = pd.to_numeric(support["satisfaction_score"], errors = "coerce")

usage["usage_count"] = pd.to_numeric(usage["usage_count"], errors = "coerce")
usage["usage_duration_secs"] = pd.to_numeric(usage["usage_duration_secs"], errors = "coerce")
usage["error_count"] = pd.to_numeric(usage["error_count"], errors = "coerce")

churn["refund_amount_usd"] = pd.to_numeric(churn["refund_amount_usd"], errors = "coerce")

#Removing duplicate rows from each dataset
accounts = accounts.drop_duplicates()
subscriptions = subscriptions.drop_duplicates()
support = support.drop_duplicates()
churn = churn.drop_duplicates()
usage = usage.drop_duplicates()

#Creating one fixed analysis date from the latest date available in the datasets
analysis_date = max(
    accounts["signup_date"].max(),
    subscriptions["start_date"].max(),
    subscriptions["end_date"].max(),
    support["submitted_at"].max(),
    support["closed_at"].max(),
    churn["churn_date"].max(),
    usage["usage_date"].max()
)

print("Cleaning complete.")
print("Analysis date:", analysis_date)

Cleaning complete.
Analysis date: 2024-12-31 19:00:00


### 4. SUBSCRIPTION FEATURE ENGINEERING

In [19]:
#Replacing missing end_dates with the analysis_date because missing end_date means the subscription is still active
subscriptions["effective_end_date"] = subscriptions["end_date"].fillna(analysis_date)

#Calculating subscription tenure in days
subscriptions["tenure_days"] = (subscriptions["effective_end_date"] - subscriptions["start_date"]).dt.days

#Removing impossible negative tenure values if they exist
subscriptions = subscriptions[subscriptions["tenure_days"] >= 0].copy()

#Estimating subscription revenue using mrr_amount and tenure
subscriptions["estimated_revenue"] = subscriptions["mrr_amount"] * (subscriptions["tenure_days"] / 30)

#Sorting by start_date so "last" means latest subscription record per account
subscriptions = subscriptions.sort_values(["account_id", "start_date"])

#Aggregating subscription-level data into account-level features
subs_agg = subscriptions.groupby("account_id").agg(
    subscription_count=("subscription_id", "count"),
    avg_mrr=("mrr_amount", "mean"),
    total_estimated_revenue=("estimated_revenue", "sum"),
    max_tenure_days=("tenure_days", "max"),
    latest_plan_tier=("plan_tier", "last"),
    latest_billing_frequency=("billing_frequency", "last"),
    latest_auto_renew_flag=("auto_renew_flag", "last"),
    had_upgrade=("upgrade_flag", "max"),
    had_downgrade=("downgrade_flag", "max"),
    subscription_churn_flag=("churn_flag", "max")
).reset_index()

print("Subscription features created.")
print(subs_agg.head())

Subscription features created.
  account_id  subscription_count      avg_mrr  total_estimated_revenue  \
0   A-00bed1                  10  3350.600000            223491.533333   
1   A-00cac8                   9  1569.000000             60961.533333   
2   A-0158bb                   6   678.333333              5897.800000   
3   A-016043                  11  1501.454545             34629.633333   
4   A-019782                   9   928.111111             89959.700000   

   max_tenure_days latest_plan_tier latest_billing_frequency  \
0              411       Enterprise                   annual   
1              472              Pro                   annual   
2              215              Pro                   annual   
3              151            Basic                  monthly   
4              572       Enterprise                   annual   

   latest_auto_renew_flag  had_upgrade  had_downgrade  subscription_churn_flag  
0                   False         True          False     

### 5. SUPPORT FEATURE ENGINEERING

In [20]:
#Creating high_priority_flag column for urgent/high tickets
support["high_priority_flag"] = support["priority"].isin(["high", "urgent"]).astype(int)

#Aggregating support tickets into account-level friction signals
support_agg = support.groupby("account_id").agg(
    ticket_count=("ticket_id", "count"),
    avg_resolution_time_hours=("resolution_time_hours", "mean"),
    avg_first_response_minutes=("first_response_time_minutes", "mean"),
    avg_satisfaction_score=("satisfaction_score", "mean"),
    escalation_rate=("escalation_flag", "mean"),
    high_priority_ticket_count=("high_priority_flag", "sum")
).reset_index()

print("Support features created.")
print(support_agg.head())

Support features created.
  account_id  ticket_count  avg_resolution_time_hours  \
0   A-00bed1             4                  31.750000   
1   A-00cac8             2                  33.000000   
2   A-0158bb             1                  32.000000   
3   A-016043             3                  30.333333   
4   A-019782             2                  10.000000   

   avg_first_response_minutes  avg_satisfaction_score  escalation_rate  \
0                      106.25                     4.0              0.0   
1                      120.50                     NaN              0.0   
2                       50.00                     3.0              0.0   
3                       78.00                     4.0              0.0   
4                      107.00                     3.0              0.0   

   high_priority_ticket_count  
0                           4  
1                           2  
2                           1  
3                           2  
4                         

### 6. USAGE FEATURE ENGINEERING

In [21]:
#Usage does NOT have account_id, so we bring account_id from subscriptions using subscription_id
usage_with_accounts = usage.merge(subscriptions[["subscription_id", "account_id"]], on = "subscription_id", how = "left")

#Checking if any usage rows failed to match to an account
missing_usage_accounts = usage_with_accounts["account_id"].isna().sum()
print("Usage rows without account_id after merge:", missing_usage_accounts)

#Removing unmatched usage rows if any exist
usage_with_accounts = usage_with_accounts.dropna(subset = ["account_id"]).copy()

#Aggregating usage into account-level engagement signals
usage_agg = usage_with_accounts.groupby("account_id").agg(
    total_usage_events=("usage_id", "count"),
    total_usage_count=("usage_count", "sum"),
    total_usage_duration_secs=("usage_duration_secs", "sum"),
    active_days=("usage_date", "nunique"),
    feature_diversity=("feature_name", "nunique"),
    total_errors=("error_count", "sum"),
    beta_feature_usage_rate=("is_beta_feature", "mean"),
    last_usage_date=("usage_date", "max")
).reset_index()

#Calculating days since last activity
usage_agg["days_since_last_activity"] = (analysis_date - usage_agg["last_usage_date"]).dt.days

print("Usage features created.")
print(usage_agg.head())

Usage rows without account_id after merge: 0
Usage features created.
  account_id  total_usage_events  total_usage_count  \
0   A-00bed1                  51                514   
1   A-00cac8                  58                602   
2   A-0158bb                  36                364   
3   A-016043                  47                490   
4   A-019782                  55                562   

   total_usage_duration_secs  active_days  feature_diversity  total_errors  \
0                     143734           49                 32            27   
1                     171366           57                 30            31   
2                     122051           35                 19            22   
3                     132075           46                 26            21   
4                     160848           55                 28            30   

   beta_feature_usage_rate last_usage_date  days_since_last_activity  
0                 0.039216      2024-12-14                  

### 7. CHURN LABEL ENGINEERING

In [22]:
#Creating churn indicator from churn events
churn["churn_event_flag"] = 1

#Sorting churn events so the last reason/date is the latest churn event
churn = churn.sort_values(["account_id", "churn_date"])

#Aggregating churn events into account-level churn signals
churn_agg = churn.groupby("account_id").agg(
    churn_event_flag = ("churn_event_flag", "max"),
    churn_event_count = ("churn_event_id", "count"),
    latest_churn_date = ("churn_date", "last"),
    churn_reason = ("reason_code", "last"),
    total_refund_amount = ("refund_amount_usd", "sum"),
    had_reactivation = ("is_reactivation", "max")
).reset_index()

print("Churn features created.")
print(churn_agg.head())

Churn features created.
  account_id  churn_event_flag  churn_event_count latest_churn_date  \
0   A-00bed1                 1                  1        2024-01-03   
1   A-016043                 1                  1        2024-08-11   
2   A-029f69                 1                  3        2024-12-14   
3   A-02cd81                 1                  1        2024-04-30   
4   A-02fac6                 1                  1        2024-08-21   

  churn_reason  total_refund_amount  had_reactivation  
0      unknown                21.17             False  
1   competitor                84.75             False  
2   competitor                32.73             False  
3      pricing                 0.00             False  
4      support                 0.00             False  


### 8. CUSTOMER 360 TABLE

In [23]:
#Starting with accounts because it has one row per customer/account
customer_360 = accounts.copy()

#Merging subscription features
customer_360 = customer_360.merge(subs_agg, on = "account_id", how = "left")

#Merging support features
customer_360 = customer_360.merge(support_agg, on = "account_id", how = "left")

#Merging usage features
customer_360 = customer_360.merge(usage_agg, on = "account_id", how = "left")

#Merging churn event features
customer_360 = customer_360.merge(churn_agg, on = "account_id", how = "left")

#Filling missing numeric values created by accounts with no support/usage/churn records
numeric_fill_cols = [
    "subscription_count",
    "avg_mrr",
    "total_estimated_revenue",
    "max_tenure_days",
    "ticket_count",
    "avg_resolution_time_hours",
    "avg_first_response_minutes",
    "escalation_rate",
    "high_priority_ticket_count",
    "total_usage_events",
    "total_usage_count",
    "total_usage_duration_secs",
    "active_days",
    "feature_diversity",
    "total_errors",
    "beta_feature_usage_rate",
    "days_since_last_activity",
    "churn_event_flag",
    "churn_event_count",
    "total_refund_amount"
]

for col in numeric_fill_cols:
    if col in customer_360.columns:
        customer_360[col] = customer_360[col].fillna(0)

#Building final churn label using both accounts churn_flag and churn events
customer_360["final_churn_flag"] = (
    (customer_360["churn_flag"] == True) |
    (customer_360["subscription_churn_flag"] == True) |
    (customer_360["churn_event_flag"] == 1)
).astype(int)

#Replacing missing churn reason with "not_churned"
customer_360["churn_reason"] = customer_360["churn_reason"].fillna("not_churned")

print("Customer 360 table created.")
print("Shape:", customer_360.shape)
print(customer_360.head())

Customer 360 table created.
Shape: (500, 42)
  account_id account_name    industry country signup_date referral_source  \
0   A-2e4581    Company_0      EdTech      US  2024-10-16         partner   
1   A-43a9e3    Company_1     FinTech      IN  2023-08-17           other   
2   A-0a282f    Company_2    DevTools      US  2024-08-27         organic   
3   A-1f0ac7    Company_3  HealthTech      UK  2023-08-27           other   
4   A-ce550d    Company_4  HealthTech      US  2024-10-27           event   

    plan_tier  seats  is_trial  churn_flag  ...  beta_feature_usage_rate  \
0       Basic      9     False       False  ...                 0.072727   
1       Basic     18     False        True  ...                 0.057143   
2       Basic      1     False       False  ...                 0.060241   
3       Basic     24      True       False  ...                 0.121951   
4  Enterprise     35     False        True  ...                 0.068966   

   last_usage_date  days_since_last

### 9. CHURN ANALYSIS FEATURE

In [24]:
#Avoiding division by zero by replacing 0 tenure with 1 where needed
safe_tenure_months = (customer_360["max_tenure_days"] / 30).replace(0, 1)

#Tickets per month normalizes ticket volume by customer lifetime
customer_360["tickets_per_month"] = customer_360["ticket_count"] / safe_tenure_months

#Revenue per day shows customer value intensity
customer_360["revenue_per_day"] = customer_360["total_estimated_revenue"] / customer_360["max_tenure_days"].replace(0, 1)

#Error rate shows product friction relative to usage volume
customer_360["error_rate"] = customer_360["total_errors"] / customer_360["total_usage_count"].replace(0, 1)

#Simple engagement score using ranked behavior metrics
customer_360["engagement_score"] = (
    customer_360["total_usage_count"].rank(pct = True) +
    customer_360["active_days"].rank(pct = True) +
    customer_360["feature_diversity"].rank(pct = True)
) / 3

#Create engagement segment
customer_360["engagement_segment"] = pd.cut(
    customer_360["engagement_score"],
    bins = [0, 0.33, 0.66, 1],
    labels = ["Low Engagement", "Medium Engagement", "High Engagement"],
    include_lowest = True
)

print("Churn analysis features created.")
print(customer_360[[
    "account_id",
    "final_churn_flag",
    "engagement_score",
    "engagement_segment",
    "tickets_per_month",
    "error_rate"
]].head())

Churn analysis features created.
  account_id  final_churn_flag  engagement_score engagement_segment  \
0   A-2e4581                 1          0.541667  Medium Engagement   
1   A-43a9e3                 1          0.206000     Low Engagement   
2   A-0a282f                 1          0.944000    High Engagement   
3   A-1f0ac7                 1          0.303667     Low Engagement   
4   A-ce550d                 1          0.703333    High Engagement   

   tickets_per_month  error_rate  
0           0.833333    0.071028  
1           0.192308    0.039437  
2           1.046512    0.058465  
3           0.152672    0.054974  
4           3.281250    0.053541  


### 10. BASIC CHURN INSIGHTS

In [25]:
#Overall churn rate
overall_churn_rate = customer_360["final_churn_flag"].mean()

print("Overall churn rate:", round(overall_churn_rate * 100, 2), "%")

#Churn by plan tier
churn_by_plan = customer_360.groupby("plan_tier")["final_churn_flag"].mean().sort_values(ascending = False) * 100
print("\nChurn rate by account plan tier:")
print(churn_by_plan.round(2))

#Churn by latest subscription plan
churn_by_latest_plan = customer_360.groupby("latest_plan_tier")["final_churn_flag"].mean().sort_values(ascending = False) * 100
print("\nChurn rate by latest subscription plan:")
print(churn_by_latest_plan.round(2))

#Churn by engagement segment
churn_by_engagement = customer_360.groupby("engagement_segment")["final_churn_flag"].mean() * 100
print("\nChurn rate by engagement segment:")
print(churn_by_engagement.round(2))

#Churn by billing frequency
churn_by_billing = customer_360.groupby("latest_billing_frequency")["final_churn_flag"].mean().sort_values(ascending = False) * 100
print("\nChurn rate by billing frequency:")
print(churn_by_billing.round(2))

#Top churn reasons
churn_reasons = customer_360[customer_360["final_churn_flag"] == 1]["churn_reason"].value_counts()
print("\nTop churn reasons:")
print(churn_reasons)

Overall churn rate: 90.0 %

Churn rate by account plan tier:
plan_tier
Pro           90.45
Basic         89.88
Enterprise    89.61
Name: final_churn_flag, dtype: float64

Churn rate by latest subscription plan:
latest_plan_tier
Basic         94.48
Pro           89.51
Enterprise    86.29
Name: final_churn_flag, dtype: float64

Churn rate by engagement segment:
engagement_segment
Low Engagement       84.38
Medium Engagement    91.33
High Engagement      94.01
Name: final_churn_flag, dtype: float64

Churn rate by billing frequency:
latest_billing_frequency
annual     90.04
monthly    89.95
Name: final_churn_flag, dtype: float64

Top churn reasons:
churn_reason
not_churned    98
features       67
support        62
budget         62
competitor     56
unknown        55
pricing        50
Name: count, dtype: int64


### 11. EXPORTING THE CLEAN DATA

In [26]:
#Exporting the final customer-level dataset for Power BI

customer_360.to_csv("ravenstack_customer_360.csv", index = False)
print("Export complete: ravenstack_customer_360.csv")

Export complete: ravenstack_customer_360.csv
